# 07 — Laya as the decision engine (self-hosted, Jev-compatible)

[Laya](https://huggingface.co/convaiinnovations/laya) is an open-weight (Apache 2.0),
**self-hosted** analog to notebook 04's Jev: same idea (typed `choice`/`score`/`noul`
questions over text, calibrated confidence, non-autoregressive — it scores predefined
answers, never generates text), same training method name (RLCD), and its HTTP server
(`laya.serve`) speaks the *exact same* `POST /v1/systemone` wire protocol as TypeSafe's
hosted Jev API. Several community projects call it "the open-source Jev-compatible
System-1 decision model" outright. Unlike Jev, there is no API key, no early access,
and no per-call cost — it runs locally via `pip install laya` (PyTorch + transformers
under the hood; the base checkpoint is a fine-tuned ModernBERT-large, i.e. it already
**is** a BERT-family model — there is no separate "self-hosted BERT" step, and no
Ollama involved: Laya is a non-generative classifier head, not a chat model, so it
does not run through Ollama at all).

**Checkpoint used here: `convaiinnovations/laya` (base, English, 421M params) — zero-shot,
same as Jev in notebook 04.** Deliberately *not* `convaiinnovations/laya-typed-decisions`:
that checkpoint is fine-tuned on four unrelated synthetic workflows (agent-trace
observability, customer service, invoice processing, security incidents) and its own
model card warns "expect it to behave like the base laya checkpoint, or worse, on
anything else" — i.e. no reason to expect it helps our ollama-vs-openai task, and a real
chance it hurts. `convaiinnovations/laya-multilingual` isn't needed either — our eval set
is English-only.

**Set expectations honestly**: Laya's own base-checkpoint zero-shot accuracy on its own
"typed-decisions" benchmark is 0.362 (near chance, 3-ish classes) — fine-tuning is what
gets it to 0.766 on that specific benchmark. Our task is only binary and worded very
concretely (same `criteria` text as Jev's notebook 04), which may fare better, but do not
assume it will — a low accuracy here is a legitimate finding about zero-shot Laya on this
task, not a bug to paper over, exactly like RouteLLM's misalignment in notebook 03.

After `decide_route` returns `ollama` or `openai`, dispatch is dumb — no regex, no
LiteLLM pick, same discipline as notebook 04.

Model card: https://huggingface.co/convaiinnovations/laya · Repo: https://github.com/NandhaKishorM/laya

Kernel: Python 3.10+. GPU (CUDA or Apple `mps`) recommended but not required — CPU works,
just slower (documented ~33ms on GPU vs ~200-450ms on CPU per question).


In [1]:
# Downloads the laya package: torch + transformers 5.x under the hood.
# First Router() call below downloads the ~421M-parameter checkpoint from the Hub
# (a few hundred MB to ~1.6GB depending on precision) and caches it locally.
%pip install laya python-dotenv -q


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd()
poc_dir = None
root = None
for p in [cwd, *cwd.parents]:
    if (p / "eval_queries.py").exists():
        poc_dir = p
        break
    if (p / "poc" / "eval_queries.py").exists():
        poc_dir = p / "poc"
        break
if poc_dir is None:
    raise FileNotFoundError("eval_queries.py not found — run from route-chatbot/ or route-chatbot/poc/")
sys.path.insert(0, str(poc_dir))

for p in [cwd, *cwd.parents]:
    if (p / ".env").exists() and (p / "main.py").exists():
        root = p
        load_dotenv(p / ".env")
        break
else:
    load_dotenv()

from eval_queries import EVAL_QUERIES

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3")
OLLAMA_MODEL_2 = os.getenv("OLLAMA_MODEL_2", "llama3")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-3.5-turbo")
TYPESAFE_API_KEY = os.getenv("TYPESAFE_API_KEY", "")

print("poc_dir", poc_dir)
print("eval queries", len(EVAL_QUERIES))
print("ollama", OLLAMA_BASE_URL, OLLAMA_MODEL, "| alt", OLLAMA_MODEL_2)
print("openai model", OPENAI_MODEL, "| key set", bool(OPENAI_API_KEY))


poc_dir /Users/tushar/PravarAI/route-chatbot/poc
eval queries 36
ollama http://localhost:11434 llama3.1:8b | alt qwen3.5:9b
openai model gpt-3.5-turbo | key set True


## Device check (informational — Laya runs on CPU, CUDA, or Apple `mps`)


In [3]:
try:
    import torch

    if torch.cuda.is_available():
        device = "cuda"
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"
    print("torch", torch.__version__, "| device laya will likely pick:", device)
except ImportError:
    print("torch not importable yet — will be pulled in by `pip install laya` above.")


torch 2.14.0 | device laya will likely pick: mps


## Load Laya, then define the same `Choice` question as Jev (notebook 04)

Same `instructions` and `criteria` text as notebook 04's `ROUTE_QUESTION`, so any
accuracy difference between the two notebooks reflects the classifier, not the prompt.


In [4]:
from laya import Router

try:
    laya_router = Router()  # downloads/caches convaiinnovations/laya on first call
except Exception as e:
    raise RuntimeError(
        "Laya failed to initialize. Need `pip install laya`, a working torch install, "
        "and network access for the first checkpoint download. Not faking a route. "
        f"Original error: {type(e).__name__}: {e}"
    ) from e

ROUTE_QUESTION = {
    "type": "choice",
    "instructions": (
        "Pick which model should answer the user message. "
        "Choose exactly one option."
    ),
    "criteria": {
        "ollama": (
            "Greetings, small talk, thanks, casual chat, or generic questions "
            "a small local model can handle."
        ),
        "openai": (
            "Coding, algorithms, debugging, analysis, design, or creative writing "
            "that needs a stronger model."
        ),
    },
}

print("laya router ready")


laya router ready


In [5]:
def decide_route(message: str):
    """Laya first — no other router runs before this. Returns (label, details)."""
    result = laya_router.predict(message, {"route": ROUTE_QUESTION})
    answer = result["answers"]["route"]
    choice = answer.get("choice")
    confidence = answer.get("confidence")
    served_model = (result.get("routing") or {}).get("model")
    if choice not in ("ollama", "openai"):
        raise RuntimeError(f"unexpected Laya choice: {choice!r}")
    return choice, {"confidence": confidence, "served_model": served_model}


## Eval (each row is one local forward pass — ~33ms on GPU, slower on CPU, no network
call once the checkpoint is cached, no per-call cost either way)


In [6]:
rows = []
for item in EVAL_QUERIES:
    t0 = time.perf_counter()
    err = None
    predicted = None
    extra = None
    try:
        predicted, extra = decide_route(item["message"])
    except Exception as e:
        err = f"{type(e).__name__}: {e}"
    ms = (time.perf_counter() - t0) * 1000
    rows.append({
        "message": item["message"],
        "expected": item["expected"],
        "predicted": predicted,
        "match": predicted == item["expected"],
        "latency_ms": round(ms, 1),
        "error": err,
        "extra": extra,
    })

n = len(rows)
ok = sum(1 for r in rows if r["match"])
errs = sum(1 for r in rows if r["error"])
mean_ms = sum(r["latency_ms"] for r in rows) / n if n else 0
print(f"accuracy {ok}/{n} ({100 * ok / n:.0f}%)  mean latency {mean_ms:.1f} ms  errors {errs}")
print()
for r in rows:
    flag = "OK  " if r["match"] else "MISS"
    extra = f"  {r['extra']}" if r["extra"] else ""
    err = f"  ERR {r['error']}" if r["error"] else ""
    print(f"  [{flag}] {r['latency_ms']:7.1f} ms  exp={r['expected']:7} pred={r['predicted']}  {r['message'][:60]}{extra}{err}")


/Users/tushar/PravarAI/pravar-monorepo/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1863.64it/s]
/Users/tushar/PravarAI/pravar-monorepo/backend/.venv/lib/python3.12/site-packages/laya/router.py:260: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent = Agent(repo, device=self.device, token=self.token, subfolder=sub)


accuracy 22/36 (61%)  mean latency 185.8 ms  errors 0

  [MISS]  5200.6 ms  exp=ollama  pred=openai  hi there  {'confidence': 0.0044, 'served_model': 'english'}
  [OK  ]    43.8 ms  exp=ollama  pred=ollama  hello  {'confidence': 0.0155, 'served_model': 'english'}
  [OK  ]    40.9 ms  exp=ollama  pred=ollama  hey  {'confidence': 0.0029, 'served_model': 'english'}
  [OK  ]    40.3 ms  exp=ollama  pred=ollama  good morning  {'confidence': 0.0716, 'served_model': 'english'}
  [OK  ]    40.4 ms  exp=ollama  pred=ollama  thanks  {'confidence': 0.0001, 'served_model': 'english'}
  [OK  ]    41.6 ms  exp=ollama  pred=ollama  how are you  {'confidence': 0.317, 'served_model': 'english'}
  [OK  ]    42.5 ms  exp=openai  pred=openai  write a function to reverse a linked list  {'confidence': 0.0938, 'served_model': 'english'}
  [OK  ]    41.7 ms  exp=openai  pred=openai  compare merge sort and quick sort  {'confidence': 0.1001, 'served_model': 'english'}
  [OK  ]    42.4 ms  exp=openai  pred=opena

## Notes (fill during the experiment)

- Compare directly to notebook 04 (Jev): same question text, same eval set, hosted vs
  self-hosted. Accuracy differences are about the classifier itself; latency differences
  are about network-hosted-API vs local-forward-pass, and are the real "router tax" story.
- `eval_queries.py`'s keyword-free stress rows (added for notebook 06) are a good read here
  too — do they expose whether Laya is doing more than surface keyword matching?
- If accuracy lands near the base checkpoint's own documented zero-shot number (0.362 on
  their benchmark), that is a legitimate result about zero-shot Laya on this task, not a
  bug — do not tune `criteria` wording to chase a better number without saying so.
- `laya.serve` (the `laya[serve]` extra) exposes this exact router over HTTP on the same
  `POST /v1/systemone` wire protocol as Jev. That means LiteLLM Auto Router v2's
  `classifier_type: jev` (found while building notebook 06 — see its intro) could point at
  a self-hosted `laya.serve` instance instead of a TypeSafe API key. Not built here — worth
  a follow-up notebook if Laya proves useful.
- Device the run actually used matters a lot for the latency number — check the device-check
  cell's output above before comparing latencies across machines/runs.


In [7]:
GENERATE = True  # flip to True during the experiment, not the scaffold

def call_ollama(message: str, model: str | None = None) -> str:
    import json
    import urllib.request

    payload = json.dumps({
        "model": model or OLLAMA_MODEL,
        "prompt": message,
        "stream": False,
    }).encode()
    req = urllib.request.Request(
        f"{OLLAMA_BASE_URL.rstrip('/')}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.loads(resp.read().decode()).get("response", "").strip()


def call_openai(message: str) -> str:
    import json
    import urllib.request

    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY is missing")
    payload = json.dumps({
        "model": OPENAI_MODEL,
        "messages": [{"role": "user", "content": message}],
        "max_tokens": 64,
    }).encode()
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions",
        data=payload,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {OPENAI_API_KEY}",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read().decode())
        return data["choices"][0]["message"]["content"].strip()


if GENERATE:
    for item in EVAL_QUERIES[:]:
        decision = decide_route(item["message"])
        label = decision[0] if isinstance(decision, tuple) else decision
        fn = call_openai if label == "openai" else call_ollama
        print("---", item["message"], "->", label)
        print(fn(item["message"])[:400])
        print()
else:
    print("GENERATE is False — routing only. Flip it to actually call models.")


--- hi there -> openai
Hello! How can I assist you today?

--- hello -> ollama
Hello! Is there something I can help you with or would you like to chat?

--- hey -> ollama
How's it going? Is there something I can help you with or would you like to chat?

--- good morning -> ollama
Good morning! Hope you're having a great start to the day! How can I help you today?

--- thanks -> ollama
You're welcome! It was a pleasure chatting with you, even if it was just a brief "thanks"! If you need anything or want to chat, feel free to start a conversation anytime. Have a great day!

--- how are you -> ollama
I'm just a computer program, so I don't have feelings or emotions like humans do. I'm functioning properly and ready to help with any questions or tasks you have. How can I assist you today?

--- write a function to reverse a linked list -> openai
Here is a sample implementation in Python for reversing a linked list:

```python
class Node:
    def __init__(self, value):
        self.value = v